# Acquirium quick start

Acquirium is a metadata + timeseries platform for water systems. The server holds two things about a plant:

1. **a semantic model** — equipment, piping, sensors and units, described with the ASHRAE 223 / WaTr ontologies
2. **the timeseries** of every measured point

The key idea: **you don't query tag names, you query meaning.** You describe what you want in domain terms — a pump, salt concentration, whatever is upstream of the RO — and acquirium finds the points. Code written this way is not specific to a plant: it runs on any plant that has a model.

To get support reach out to [Mete](mailto:saka@mines.edu)!

## Setup
0. Follow the steps in [deployments/WATERTAP/README.md](../../deployments/WATERTAP/README.md) to install requirements
1. Start the server with a config: `acquirium server --config deployments/WATERTAP/scripts/acquirium.toml`
2. Connect:

In [6]:
from acquirium import Acquirium
acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)

########################################################################
########################################################################
########################################################################
## For better display of polars dataframes in Jupyter notebooks
import polars as pl
pl.Config.set_tbl_width_chars(1000)
pl.Config.set_fmt_str_lengths(200)

polars.config.Config

## Find entities by ontology classes
`_class` takes plain text. The server resolves it against the ontology, so you don't need to know any ontology URIs to explore:

In [7]:
q = acq.find_entity(_class="Pump", alias="pump")
q.metadata()

pump
str
"""wbs:P1"""
"""wbs:P2"""
"""wbs:intake"""


## A query is a description, data comes last
Queries are built step by step and nothing is fetched until you ask. `.metadata()` tells you *which points* matched; `.dataframe()` then pulls the numbers. Let's ask for every salt concentration in the plant:

In [10]:
q = (acq.find_all_data()
        .filter_by_substance("constituent salt")
        .filter_by_quantity_kind("mass concentration"))
q.metadata().unique()

0
str
"""wbs:storage-tank-3-out-tds-concentration"""
"""wbs:PXR-brine-out-tds-concentration"""
"""wbs:intake-in-tds-concentration"""


Three points matched: 
- seawater feed
- brine discharge 
- product water 

We can access to their timeseries values with:

In [11]:
q.dataframe(shape="wide", cast_value="float").drop_nulls().tail(3)

time,wbs:PXR-brine-out-tds-concentration,wbs:intake-in-tds-concentration,wbs:storage-tank-3-out-tds-concentration
"datetime[μs, UTC]",f64,f64,f64
2026-07-03 07:00:09.059617 UTC,63.225117,35.713423,0.23958
2026-07-03 07:00:23.674013 UTC,62.75194,36.29365,0.239937
2026-07-03 07:00:44.597356 UTC,62.729668,36.29293,0.239704


## The topology is queryable
For questions about *where* such as what feeds this unit, what is measured downstream of that one, etc.; We can explore and find the related equipment. For instance, let's find the RO membraine and it's upstream pump:

In [12]:
(acq.find_entity(_class="reverse osmosis membrane", alias="ro")
    .find_related(_class="Pump", alias="feed_pump", direction="upstream", hops=3)
    .metadata())

ro,feed_pump
str,str
"""wbs:RO""","""wbs:P1"""
"""wbs:RO""","""wbs:P2"""


## Units
Every property can carry a QUDT unit and Acquirium can convert these units to each other:

In [30]:
data = q.data()
print(data.units())
data.convert_to("mg/L").dataframe().drop_nulls().tail(3)

{'0': 'http://qudt.org/vocab/unit/KiloGM-PER-M3'}


time,0__wbs:PXR-brine-out-tds-concentration,0__wbs:intake-in-tds-concentration,0__wbs:storage-tank-3-out-tds-concentration
"datetime[μs, UTC]",f64,f64,f64
2026-07-03 07:09:04.614367 UTC,61983.58577,36177.016885,234.823034
2026-07-03 07:09:22.128395 UTC,62409.315797,36777.326348,240.056871
2026-07-03 07:09:37.373828 UTC,62118.02508,36195.764662,235.77807


## Where to go next
- `watertap-1.ipynb` — the client reference: every feature, plus the internals (query graph, generated SPARQL)
- `regulation.ipynb` — a real application: checking this plant against the California Ocean Plan and Title 22 drinking water standards

